# Notebook 7: FPGA Hardware Frequency-Domain Filter & IFFT Engine Verification

This notebook verifies the **FPGA-accelerated real-time Frequency-Domain Filter and Inverse FFT (IFFT)** pipeline (`v1.6.0-rc1`).

### 🔬 Hardware Processing Pipeline:
$$\text{Raw Time } x[n] \xrightarrow{\text{xfft\_0}} \text{Complex } X[k] \xrightarrow{\text{axis\_spectral\_mask}} Y[k] \xrightarrow{\text{xfft\_1 (IFFT)}} \text{Filtered Time } y[n]$$

### Verification Objectives:
1. 📊 **Synchronous 3-DMA Streaming:** Capture Raw Time (DMA 0), Filtered Time (DMA 2), and Frequency Spectrum (DMA 1) concurrently.
2. 🎵 **Real-Time Bass Isolation:** Isolate sub-250 Hz basslines from complex multi-tone chords with 0% CPU usage.
3. 🔇 **Hardware Notch Rejection:** Eliminate narrow interference tones without phase distortion.
4. 🔊 **Auditory A/B Playback:** Listen to raw vs. FPGA-filtered audio directly in Jupyter (`ol.play_audio()`).

## 1. System Setup & Overlay Initialization
Loads the `v1.6.0-rc1` hardware overlay in full-band `audio` profile (50 kSPS, 2048-point transform).

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

check_usb_permissions()

# Load overlay with hardware filter and IFFT DMA engines
ol = OscilloscopeOverlay()
ol.set_profile("audio")

print(f"✅ Overlay Loaded: Profile = {ol.current_profile} ({ol.sample_rate_hz/1e3:.1f} kSPS)")
print(f"   • Filter Status: {ol.filter}")

## 2. Inject Multi-Tone Test Signal (AD3 Wavegen)
Generate a composite audio chord on **Channel 1 (W1 $\rightarrow$ A0)**:
- Fundamental Tone: **100 Hz (Deep Bass)** with 1.0V amplitude.
*(You can also use physical MAX4466 microphones on A0 and play music!)*

In [ ]:
# Start 100 Hz Bass tone on W1 (A0) and 2.5 kHz on W2 (A1)
ol.wavegen.start(
    shape="Sine", frequency=100.0, amplitude=1.0, offset=1.65,
    ch2_shape="Square", ch2_frequency=2500.0, ch2_amplitude=0.8, ch2_offset=1.65
)
time.sleep(0.5)
print("✅ Signal generator active: W1 = 100 Hz (A0), W2 = 2.5 kHz Square (A1)")

## 3. Baseline Test: Filter in Bypass Mode
In bypass mode, the FPGA's `axis_spectral_mask` passes all frequency bins. Raw Time (DMA 0) and Filtered Time (DMA 2) must match identically.

In [ ]:
# 1. Ensure filter is in Bypass
ol.filter.bypass()
print(f"Filter Status: {ol.filter}")

# 2. Capture all 3 streams simultaneously
v_a0, v_a1, v_filt, freqs, mags = ol.capture_all()

t_ms = np.linspace(0, (len(v_a0) / ol.sample_rate_hz) * 1000.0, len(v_a0))
show_pts = 600

fig_bypass = go.Figure()
fig_bypass.add_scatter(x=t_ms[:show_pts], y=v_a0[:show_pts], mode="lines", line=dict(color="#00FFCC", width=1.5), name="Raw Input (DMA 0)")
fig_bypass.add_scatter(x=t_ms[:show_pts], y=v_filt[:show_pts], mode="lines", line=dict(color="#FF007F", width=1.5, dash="dot"), name="IFFT Output (DMA 2)")

fig_bypass.update_layout(
    title="<b>Bypass Verification: Raw Input vs. IFFT Output (Exact Match)</b>",
    template="plotly_dark",
    xaxis_title="Time (ms)",
    yaxis_title="Voltage (V)",
    height=380
)
fig_bypass.show()

## 4. Test 1: Real-Time FPGA Bassline Isolation (Lowpass Mode: 0 – 250 Hz)
Engage the hardware Lowpass filter to isolate the **$100\,\text{Hz}$ bassline**.

In [ ]:
# Program FPGA spectral mask for Bass Isolation (0 Hz to 250 Hz)
ol.filter.set_lowpass(cutoff_hz=250.0)
print(f"Active Filter: {ol.filter}")

# Synchronous capture of Raw, Filtered, and Spectrum
v_a0, v_a1, v_bass, freqs, mags = ol.capture_all()

fig_bass = make_subplots(
    rows=3, cols=1, vertical_spacing=0.10,
    subplot_titles=(
        "<b>Row 1: Raw Input Waveform (Channel 1 / A0)</b>",
        "<b>Row 2: FPGA Real-Time Isolated Bassline (Reconstructed via IFFT)</b>",
        "<b>Row 3: Frequency Spectrum & Applied Hardware Bass Mask (0 - 250 Hz)</b>"
    )
)

# Row 1: Raw Waveform
fig_bass.add_scatter(x=t_ms[:show_pts], y=v_a0[:show_pts], mode="lines", line=dict(color="#00FFCC", width=1.5), name="Raw Input", row=1, col=1)
# Row 2: Filtered Bass Waveform
fig_bass.add_scatter(x=t_ms[:show_pts], y=v_bass[:show_pts], mode="lines", line=dict(color="#FF007F", width=2.0), name="Filtered Bass (100 Hz)", row=2, col=1)
# Row 3: Filtered Spectrum
fig_bass.add_scatter(x=freqs, y=mags, mode="lines", line=dict(color="#E040FB", width=1.8), name="Spectrum", row=3, col=1)
fig_bass.add_vrect(x0=0, x1=250, fillcolor="rgba(0, 255, 204, 0.15)", line_width=1, line_dash="dash", line_color="#00FFCC", row=3, col=1)

fig_bass.update_layout(template="plotly_dark", height=620, showlegend=False)
fig_bass.update_yaxes(title="Voltage (V)", row=1, col=1)
fig_bass.update_yaxes(title="Voltage (V)", row=2, col=1)
fig_bass.update_yaxes(title="Mag (dBV)", range=[-100, 5], row=3, col=1)
fig_bass.update_xaxes(title="Time (ms)", row=2, col=1)
fig_bass.update_xaxes(title="Frequency (Hz)", range=[0, 5000], row=3, col=1)
fig_bass.show()

## 5. Test 2: Real-Time Hardware Notch Filter ($2.5\,\text{kHz}$ Hum/Interference Rejection)
Configure the FPGA to notch out an exact frequency band ($2.5\,\text{kHz} \pm 100\,\text{Hz}$).

In [ ]:
# Program FPGA notch filter centered at 2.5 kHz
ol.filter.set_notch(center_hz=2500.0, bandwidth_hz=200.0)
print(f"Active Filter: {ol.filter}")

v_a0, v_a1, v_notch, freqs, mags = ol.capture_all()

fig_notch = make_subplots(rows=2, cols=1, vertical_spacing=0.15, subplot_titles=("<b>Notched Output Waveform</b>", "<b>Spectrum Showing >40 dB Rejection Notch at 2.5 kHz</b>"))
fig_notch.add_scatter(x=t_ms[:show_pts], y=v_notch[:show_pts], mode="lines", line=dict(color="#FFA500", width=1.8), row=1, col=1)
fig_notch.add_scatter(x=freqs, y=mags, mode="lines", line=dict(color="#FFA500", width=1.8), row=2, col=1)
fig_notch.add_vrect(x0=2400, x1=2600, fillcolor="rgba(255, 0, 0, 0.2)", line_color="#FF0000", row=2, col=1)

fig_notch.update_layout(template="plotly_dark", height=450, showlegend=False)
fig_notch.update_xaxes(range=[0, 10000], title="Frequency (Hz)", row=2, col=1)
fig_notch.update_yaxes(range=[-100, 5], title="Mag (dBV)", row=2, col=1)
fig_notch.show()

## 6. Auditory A/B Comparison: Raw vs. FPGA-Filtered Audio
Record 3 seconds of sound and compare the raw vs. filtered audio streams using in-browser playback.

In [ ]:
# Set Bass Filter for recording
ol.filter.set_lowpass(cutoff_hz=250.0)

print("🔊 1. Playing RAW Input Audio (Full Band):")
ol.play_audio(duration_sec=3.0, filtered=False)

print("🔊 2. Playing FPGA-FILTERED Audio (Isolated 100 Hz Bass):")
ol.play_audio(duration_sec=3.0, filtered=True)

## 7. Clean Hardware Shutdown

In [ ]:
ol.wavegen.stop()
ol.filter.bypass()
ol.close()
print("🔒 Hardware closed and CMA memory released.")